In [121]:
#import libraries
!pip install numpy pandas matplotlib seaborn

In [122]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [123]:
# import databse / Data

In [124]:
conn = sqlite3.connect('customer_churn.db')

sql_query = """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
"""

tables = pd.read_sql(sql_query, conn)

# create dataframe for each table
for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    globals()[f"df_{table_name}"] = df
    print(f"Created dataframe: df_{table_name}")

conn.close()

Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support
Created dataframe: df_customer_churn


In [125]:
#Data_cleaning
df_db_customer.head()
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
16,0020-JDNXP,rikim,India,Meghalaya,Female,1994-08-19 00:00:00,NaN,None
17,0021-IKXGC,vishakha,India,Rajasthan,Female,2000-09-02 00:00:00,NaN,None
18,0022-TCJCI,raghvendra,India,Telangana,Male,1983-12-30 00:00:00,NaN,None
19,0023-HGHWL,rishabh,India,Uttar Pradesh,Men,1991-05-14 00:00:00,NaN,None
20,0023-UYUPN,sudevi,India,Maharashtra,Women,1977-10-06 00:00:00,NaN,None


In [126]:
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     str   
 1   name        21 non-null     str   
 2   country     18 non-null     str   
 3   state       21 non-null     str   
 4   gender      21 non-null     str   
 5   dob         21 non-null     str   
 6   interests   4 non-null      str   
 7   pincode     0 non-null      object
dtypes: object(1), str(7)
memory usage: 1.4+ KB


In [127]:
# a. Rename column - name to CUstomer_name
# b. Drop column - Intrests and Pincode
# c. Change DataType - dob
# d. Data standarditation - gender
# e. Fix missing values - Country

In [128]:
# Rename column - name to CUstomer_name

df_db_customer.rename(columns = {'name' : 'customer_name'},inplace = True)
df_db_customer.head()

,customerid,customer_name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [129]:
# b. Drop column - Intrests and Pincode

df_db_customer.drop(columns=['interests','pincode'],errors='ignore')
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerid     21 non-null     str   
 1   customer_name  21 non-null     str   
 2   country        18 non-null     str   
 3   state          21 non-null     str   
 4   gender         21 non-null     str   
 5   dob            21 non-null     str   
 6   interests      4 non-null      str   
 7   pincode        0 non-null      object
dtypes: object(1), str(7)
memory usage: 1.4+ KB


In [130]:
# c. Change DataType - dob

df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])

In [131]:
# d. Data standarditation - gender

df_db_customer['gender'] = df_db_customer['gender'].replace({'Men' : 'male','Women' : 'female'})

In [132]:
df_db_customer[df_db_customer['country'].isnull()]

,customerid,customer_name,country,state,gender,dob,interests,pincode
5,0013-MHZWF,durga,NaN,Delhi,female,1988-12-10,NaN,None
8,0015-UOCOJ,maya,NaN,Kathmandu,female,1985-07-07,NaN,None
12,0018-NYROU,chitra,NaN,Telangana,Female,2004-12-01,NaN,None


In [133]:
# e. Fix missing values - Country
# Group by state, take the first known country for that state, and fill missing values

df_db_customer['country'] = df_db_customer['country'].fillna(
    df_db_customer.groupby('state')['country'].transform('first')
)

In [134]:
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaN,NaN,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaN,NaN,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaN,NaN,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88


In [135]:
df_db_subscription.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     str    
 1   subscription_start_date  21 non-null     str    
 2   subscription_type        21 non-null     str    
 3   renewal_date             21 non-null     str    
 4   plan_type                21 non-null     str    
 5   contract_type            21 non-null     str    
 6   cancellation_date        6 non-null      str    
 7   cancellation_reason      6 non-null      str    
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), str(8)
memory usage: 1.9 KB


In [136]:
date_col = ['subscription_start_date', 'renewal_date', 'cancellation_date']

df_db_subscription[date_col] = df_db_subscription[date_col].apply(pd.to_datetime)
df_db_subscription.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               21 non-null     str           
 1   subscription_start_date  21 non-null     datetime64[us]
 2   subscription_type        21 non-null     str           
 3   renewal_date             21 non-null     datetime64[us]
 4   plan_type                21 non-null     str           
 5   contract_type            21 non-null     str           
 6   cancellation_date        6 non-null      datetime64[us]
 7   cancellation_reason      6 non-null      str           
 8   monthly_charges          21 non-null     float64       
 9   cltv                     21 non-null     int64         
 10  churn_score              21 non-null     int64         
dtypes: datetime64[us](3), float64(1), int64(2), str(5)
memory usage: 1.9 KB


In [137]:
df_db_support.head()
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      str   
 1   complaint_date  9 non-null      str   
 2   escalations     9 non-null      str   
 3   csat_score      9 non-null      int64 
 4   col_1           0 non-null      object
 5   comment         4 non-null      str   
dtypes: int64(1), object(1), str(4)
memory usage: 564.0+ bytes


In [138]:
df_db_support.drop(columns=['col_1', 'comment'], errors='ignore', inplace=True)
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   customerid      9 non-null      str  
 1   complaint_date  9 non-null      str  
 2   escalations     9 non-null      str  
 3   csat_score      9 non-null      int64
dtypes: int64(1), str(3)
memory usage: 420.0 bytes


In [139]:
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      9 non-null      str           
 1   complaint_date  9 non-null      datetime64[us]
 2   escalations     9 non-null      str           
 3   csat_score      9 non-null      int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 420.0 bytes


In [140]:
# Feature Engineering & Data Analysis

In [141]:
# create a new column using existing column - churn_flag
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(),1,0)
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score,churn_flag
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,NaN,13.99,627,12,0
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91,1
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,NaN,6.99,210,34,0
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,NaN,22.99,1725,8,0
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88,1


In [142]:
df_db_support['complaint_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')

In [143]:
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customerid', keep = 'last')

In [144]:
df_db_support = df_db_support.drop(columns=['col_1', 'comment'], errors='ignore')

In [145]:
# joining tables

df = (df_db_subscription
    .merge(df_db_customer, on ='customerid', how = 'left')
    .merge(df_db_support, on ='customerid', how = 'left'))

In [146]:
df.shape

(21, 23)

In [147]:
df.to_csv('exported_churn_data.csv',index=False)

In [148]:
#Data_Analysis

In [149]:
# 1. Churn Rate

churn_rate = df['churn_flag'].mean()*100
print("churn_rate =", round(churn_rate,2),"%")

churn_rate = 28.57 %


In [150]:
# 2. retenion Rate

retenion_rate = 100 - churn_rate
print("retenion_rate =", round(retenion_rate,2),"%")

retenion_rate = 71.43 %


In [151]:
# 3. Churn By plan Type

churn_by_plan = (df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name = 'churn_rate_pct'))
print(churn_by_plan)

  plan_type  churn_rate_pct
0     Basic           60.00
1   Premium           14.29
2  Standard           22.22


In [152]:
# 5. ARPU - Avg Revenue Per User
arpu = df['monthly_charges'].mean()
print('ARUP =', round(arpu,2))

ARUP = 18.85


In [153]:
# 6. Average Customer tenure
#count of days user has used our service : cancilation date else current date

today = pd.Timestamp.today()

df['tenure_days'] = np.where(
         df['cancellation_date'].notna(),
    (df['cancellation_date'] - df['subscription_start_date']).dt.days,
    (today - df['subscription_start_date']).dt.days
)

avg_tenure = df['tenure_days'].mean()
print("Average Tenure (Days) =", round(avg_tenure, 0))

Average Tenure (Days) = 1555.0


In [154]:
# 7. Revenue at risk - revenue lost from churned users

revenue_at_risk = df.loc[df['churn_flag']==1, 'monthly_charges'].sum()
print("Revenue at Risk (Rs Lakhs) =", revenue_at_risk)

Revenue at Risk (Rs Lakhs) = 73.94


In [155]:
# 8. Esclation Rate

escalation_rate = (df['escalations']=='Y').mean()*100
print("Esclation Rate = ", round(escalation_rate, 2), "%")

Esclation Rate =  19.05 %


In [156]:

import numpy as np
import pandas as pd
import sqlite3

np.random.seed(42)  # For reproducible random generation
n_samples = 2100

# 1. Customer Demographics
customer_ids = [f"CUST-{i:04d}" for i in range(1, n_samples + 1)]

names = [
    'Aarav', 'Ananya', 'Rohan', 'Priya', 'Karan', 'Sneha', 'Vikram', 'Neha', 
    'Rahul', 'Pooja', 'Amit', 'Riya', 'Sanjay', 'Divya', 'Arjun', 'Meera',
    'Rajesh', 'Sunita', 'Suresh', 'Anita', 'Manish', 'Kavita', 'Deepak', 'Swati'
]
customer_names = np.random.choice(names, size=n_samples)

state_country_map = {
    'Maharashtra': 'India', 'Karnataka': 'India', 'Delhi': 'India', 
    'Nagaland': 'India', 'Meghalaya': 'India', 'Rajasthan': 'India', 
    'Telangana': 'India', 'Uttar Pradesh': 'India', 'Kathmandu': 'Nepal'
}
states = list(state_country_map.keys())
selected_states = np.random.choice(states, size=n_samples)
selected_countries = [state_country_map[st] for st in selected_states]

genders = np.random.choice(['male', 'female'], size=n_samples, p=[0.52, 0.48])

start_dob = pd.Timestamp('1975-01-01')
end_dob = pd.Timestamp('2003-12-31')
dobs = [start_dob + pd.Timedelta(days=int(np.random.randint(0, (end_dob - start_dob).days))) for _ in range(n_samples)]

# 2. Subscription & Contract Details
plan_types = np.random.choice(['Basic', 'Standard', 'Premium'], size=n_samples, p=[0.40, 0.35, 0.25])
contract_types = np.random.choice(['Monthly', 'Annual'], size=n_samples, p=[0.60, 0.40])
subscription_types = np.random.choice(['Paid', 'Refferal', 'Organic'], size=n_samples, p=[0.50, 0.30, 0.20])

# Realistic pricing based on plan type
monthly_charges = []
for plan in plan_types:
    if plan == 'Basic':
        monthly_charges.append(round(np.random.uniform(6.99, 11.99), 2))
    elif plan == 'Standard':
        monthly_charges.append(round(np.random.uniform(12.99, 18.99), 2))
    else:  # Premium
        monthly_charges.append(round(np.random.uniform(19.99, 29.99), 2))

# Dates
start_sub = pd.Timestamp('2020-01-01')
end_sub = pd.Timestamp('2024-06-30')
sub_start_dates = [start_sub + pd.Timedelta(days=int(np.random.randint(0, (end_sub - start_sub).days))) for _ in range(n_samples)]
renewal_dates = [dt + pd.Timedelta(days=365) for dt in sub_start_dates]

# 3. Support & Escalation Logic
escalations = np.random.choice(['Y', 'N'], size=n_samples, p=[0.25, 0.75])
csat_scores = []
complaint_counts = []

for esc in escalations:
    if esc == 'Y':
        csat_scores.append(np.random.choice([1, 2, 3], p=[0.50, 0.35, 0.15]))
        complaint_counts.append(np.random.randint(2, 6))
    else:
        csat_scores.append(np.random.choice([3, 4, 5], p=[0.20, 0.45, 0.35]))
        complaint_counts.append(np.random.choice([0, 1], p=[0.70, 0.30]))

# 4. Realistic Churn Probability Model
churn_flags = []
cancellation_dates = []
cancellation_reasons = []

reasons = ['Too expensive', 'Switched to competitor', 'Service outages', 'Poor customer support', 'Lack of features']

today_dt = pd.Timestamp.today()

for i in range(n_samples):
    # Base risk factor
    risk = 0.15
    if plan_types[i] == 'Basic': risk += 0.20
    if contract_types[i] == 'Monthly': risk += 0.15
    if escalations[i] == 'Y': risk += 0.25
    if csat_scores[i] <= 2: risk += 0.20
    
    # Cap probability between 5% and 85%
    prob = min(max(risk, 0.05), 0.85)
    churn = np.random.choice([1, 0], p=[prob, 1 - prob])
    churn_flags.append(churn)
    
    if churn == 1:
        # Cancellation date between subscription start and today
        max_days = (today_dt - sub_start_dates[i]).days
        if max_days > 30:
            canc_days = np.random.randint(30, max_days)
        else:
            canc_days = 15
        canc_dt = sub_start_dates[i] + pd.Timedelta(days=canc_days)
        cancellation_dates.append(canc_dt)
        cancellation_reasons.append(np.random.choice(reasons))
    else:
        cancellation_dates.append(pd.NaT)
        cancellation_reasons.append(None)

# 5. Tenure and CLTV
tenure_days = []
for i in range(n_samples):
    if pd.notna(cancellation_dates[i]):
        tenure_days.append((cancellation_dates[i] - sub_start_dates[i]).days)
    else:
        tenure_days.append((today_dt - sub_start_dates[i]).days)

cltv = [int(monthly_charges[i] * (tenure_days[i] / 30.44)) for i in range(n_samples)]
churn_scores = [np.random.randint(70, 100) if churn_flags[i] == 1 else np.random.randint(1, 45) for i in range(n_samples)]

# Create Final DataFrame
df_scaled = pd.DataFrame({
    'customerid': customer_ids,
    'customer_name': customer_names,
    'country': selected_countries,
    'state': selected_states,
    'gender': genders,
    'dob': dobs,
    'subscription_start_date': sub_start_dates,
    'subscription_type': subscription_types,
    'renewal_date': renewal_dates,
    'plan_type': plan_types,
    'contract_type': contract_types,
    'cancellation_date': cancellation_dates,
    'cancellation_reason': cancellation_reasons,
    'monthly_charges': monthly_charges,
    'cltv': cltv,
    'churn_score': churn_scores,
    'churn_flag': churn_flags,
    'escalations': escalations,
    'csat_score': csat_scores,
    'complaint_count': complaint_counts,
    'tenure_days': tenure_days
})

# Save scaled dataset to CSV and SQLite
df_scaled.to_csv('customer_churn_scaled.csv', index=False)
df = df_scaled
conn_scaled = sqlite3.connect('customer_churn_scaled.db')
df_scaled.to_sql('customer_churn_analytics', conn_scaled, if_exists='replace', index=False)
conn_scaled.close()

print(f"✅ SUCCESS: Dataset successfully scaled to {df_scaled.shape[0]} rows and {df_scaled.shape[1]} columns!")
print(f"📊 Scaled Churn Rate: {round(df_scaled['churn_flag'].mean() * 100, 2)}%")
print(f"💰 Scaled Revenue at Risk: ${round(df_scaled[df_scaled['churn_flag']==1]['monthly_charges'].sum(), 2):,}")

✅ SUCCESS: Dataset successfully scaled to 2100 rows and 21 columns!
📊 Scaled Churn Rate: 44.19%
💰 Scaled Revenue at Risk: $13,234.87


In [157]:
import sqlite3
import pandas as pd

# 1. Connect to your SQLite database
conn = sqlite3.connect('customer_churn_scaled.db')

# 2. Query using the correct table name: customer_churn_analytics
sql_query = """
SELECT 
    plan_type,
    COUNT(*) AS total_customers,
    SUM(churn_flag) AS churned_customers,
    ROUND(AVG(churn_flag) * 100, 2) AS churn_rate_pct,
    ROUND(AVG(monthly_charges), 2) AS arpu,
    SUM(CASE WHEN churn_flag = 1 THEN monthly_charges ELSE 0 END) AS revenue_at_risk
FROM customer_churn_analytics
GROUP BY plan_type
ORDER BY churn_rate_pct DESC;
"""

# 3. Execute and display results
df_plan_metrics = pd.read_sql_query(sql_query, conn)
print(df_plan_metrics)

  plan_type  total_customers  churned_customers  churn_rate_pct   arpu  \
0     Basic              858                474           55.24   9.46   
1   Premium              462                170           36.80  24.89   
2  Standard              780                284           36.41  16.03   

   revenue_at_risk  
0          4476.69  
1          4189.08  
2          4569.10  


In [158]:
print(df.shape)

(2100, 21)
